<a href="https://colab.research.google.com/github/AniDas10/FastSFT/blob/main/colab/FastSFT_Playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FastSFT Playground

**Teach a small, weak model to answer like a big, smart one.**

[FastSFT](https://github.com/AniDas10/FastSFT) distills a large LLM into a tiny one: it asks a big "parent" model tough questions in whatever style/persona you want, keeps only the good answers (an LLM judge filters the rest), and fine-tunes a small "child" model on them. The result is a tiny model that mimics the big one's style -- for a fraction of the size and cost.

This notebook has two parts, and you don't need to run both:

1. **Try it now** -- play with a model I've already trained (`Qwen2.5-0.5B-Instruct` tuned into a "staff software engineer" persona), zero setup, zero API keys.
2. **Make your own** -- pick any persona, generate a dataset from a big model, fine-tune your own tiny model on Colab's free GPU, and (optionally) publish both to your own Hugging Face account.

**Before you start:** grab a free GPU -- `Runtime` -> `Change runtime type` -> `T4 GPU`. Everything here works on CPU too, just slower.

> This notebook is meant to be shared. Never type an API key directly into a code cell -- the key-entry cells below use `getpass`, which doesn't echo or save what you type, so it's safe to share this notebook (and its outputs) publicly afterward.

## Setup

In [ ]:
# Installs fastsft straight from GitHub (this is an actively-evolving personal
# project -- main is more current than the last PyPI release) plus two extras:
# local-training (torch/peft/trl/accelerate, to fine-tune here on Colab's own
# GPU instead of dispatching to a cloud provider) and evaluation (adds
# sentence-transformers, for the LLM-judge + embedding-similarity scoring below).
!pip install -q "fastsft[local-training,evaluation] @ git+https://github.com/AniDas10/FastSFT.git@main"

import torch

if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)} -- training/inference will use it automatically.")
else:
    print(
        "No GPU detected -- everything below still works on CPU, just slower.\n"
        "To enable one: Runtime > Change runtime type > T4 GPU, then re-run this cell."
    )

---
## Part 1 -- Try it now (no setup, no API keys)

This loads a public adapter I've already trained -- [`anidas10/staff-eng-persona-adapter`](https://huggingface.co/anidas10/staff-eng-persona-adapter) -- fine-tuned on [`anidas10/staff-eng-persona-dataset`](https://huggingface.co/datasets/anidas10/staff-eng-persona-dataset) to answer *any* question (not just technical ones) the way a staff software engineer would: structured, systems-thinking, precise.

It generates two answers side by side: **untuned** (the plain base model) and **tuned** (the same model with the adapter applied), so you can see exactly what the fine-tuning changed.

In [ ]:
# Side-effect import: must precede distilabel/transformers (quiets their
# import-time warning noise). Keep it first in any fastsft notebook cell.
import fastsft.warnings_filter  # noqa: F401

from IPython.display import Markdown, display

from fastsft.eval.inference import ChildInferenceEngine
from fastsft.hf_helper import resolve_input

PUBLIC_ADAPTER_REPO = "anidas10/staff-eng-persona-adapter"

# Hugging Face Hub repo id -> local path to the latest pushed run, so
# ChildInferenceEngine (which expects a local PEFT adapter directory) can load it.
_public_adapter_dir = resolve_input(PUBLIC_ADAPTER_REPO, "model")
_public_engine = ChildInferenceEngine(_public_adapter_dir)


def compare(engine: ChildInferenceEngine, prompt: str) -> None:
    """Prints the untuned (base model) and tuned (adapter applied) answers to
    `prompt` side by side."""
    tuned = engine.generate_tuned([prompt])[0]
    untuned = engine.generate_untuned([prompt])[0]
    display(Markdown(
        f"### \U0001f4ac {prompt}\n\n"
        f"**\U0001f7e2 Tuned (adapter applied)**\n\n{tuned}\n\n"
        f"---\n\n"
        f"**\u26aa Untuned (base model only)**\n\n{untuned}\n\n"
        f"---"
    ))


print(f"Loaded '{PUBLIC_ADAPTER_REPO}'. Ready -- run the next cell.")

In [ ]:
# Edit this to whatever you want to ask -- any topic, not just technical ones.
compare(_public_engine, "Should I move in with my friend?")

In [ ]:
compare(_public_engine, "How is the weather today?")

### How was it trained?

FastSFT ships a training-telemetry viewer that reads the loss curve straight off the adapter directory (no API calls -- it's just the training log saved alongside the weights) and diagnoses it (learned? overfit? stopped early?).

In [ ]:
from fastsft.training.stats import load_stats
from fastsft.training.stats_viewer import render as render_stats

render_stats(load_stats(_public_adapter_dir), _public_adapter_dir)

### How does it score?

FastSFT also ships an evaluation module: an LLM judges the tuned model against its untuned base *and* the parent teacher (position-debiased -- each pair is judged both ways round), plus embedding similarity to the parent. This is a real run I precomputed (6 prompts, so treat it as a demo of the *tool*, not a final verdict -- notice how the module itself flags most of these as statistically inconclusive at this sample size, rather than overclaiming from noise).

In [ ]:
from fastsft.eval.results_viewer import render as render_eval_results

# A real evaluation run against the public adapter above (6 prompts, judged by
# deepseek/deepseek-chat against the meta-llama/llama-3.3-70b-instruct parent).
# Embedded directly so this cell needs no API key -- run live against your own
# model in Part 2 below.
_public_eval_results = {
    "run_id": "colab-demo",
    "adapter_dir": PUBLIC_ADAPTER_REPO,
    "parent_model": "meta-llama/llama-3.3-70b-instruct",
    "judge_model": "deepseek/deepseek-chat",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "num_prompts": 6,
    "swap_positions": True,
    "comparisons": {
        "tuned_vs_untuned": {"wins": 1, "ties": 4, "losses": 1, "win_rate": 0.5, "orders_judged": 2},
        "parent_likeness": {"wins": 2, "ties": 3, "losses": 1, "win_rate": 0.5833333333333334, "orders_judged": 2},
        "tuned_vs_parent": {"wins": 1, "ties": 1, "losses": 4, "win_rate": 0.25, "orders_judged": 2},
    },
    "similarity_to_parent": {
        "tuned_vs_parent": 0.7115191121896108,
        "untuned_vs_parent": 0.6926342944304148,
    },
    "samples": [],
}

render_eval_results(_public_eval_results, PUBLIC_ADAPTER_REPO)

---
## Part 2 -- Make your own

Pick any persona, generate a fresh dataset from a big "parent" model, and fine-tune your own tiny model on Colab's GPU. This is the exact same pipeline as `fastsft`'s CLI / `trial_run.py` -- just driven from notebook cells.

You'll need a **free** [OpenRouter](https://openrouter.ai) API key (Keys -> Create Key) to generate the dataset -- the "parent" and "judge" models are called through it. A Hugging Face token is optional, only needed if you want to publish your dataset/adapter to your own account.

In [ ]:
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key (required, input hidden): ")

_hf_token = getpass("Hugging Face token (optional -- leave blank to skip publishing, press Enter): ")
if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    print("Hugging Face token set -- you can publish your dataset/adapter below.")
else:
    print("No Hugging Face token -- your dataset/adapter will only be saved locally to this Colab session.")

### Describe your persona

Be explicit if you want broad topic coverage -- e.g. "any topic" / "cooking, relationships, finance, whatever" -- otherwise the guide model may read a domain-sounding phrase (like "technical") as a restriction rather than a style.

In [ ]:
# The persona: style, tone, domain (or explicitly "any domain").
PROMPT = "Answer everything related to any topic like a pirate captain -- salty, colorful, but still genuinely helpful."

# The model you're actually training. Small = fast/cheap on a free Colab GPU.
CHILD_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# Set these to "your-hf-username/repo-name" to publish to your own Hugging Face
# account (requires the HF token above); leave blank to keep everything local
# to this Colab session.
HF_DATASET_REPO_ID = ""
HF_MODEL_REPO_ID = ""

from fastsft.data.config import DataGenerationConfig

# 30 samples keeps this a ~1-2 minute, low-cost demo. Bump this up (the
# pipeline's own default is 100) for a better-trained adapter once you know
# what persona you want.
generation = DataGenerationConfig(num_samples=30)

### Run the pipeline: generate data -> format for the child model -> fine-tune

Training runs **locally** in this Colab session (on the GPU detected above, or CPU if none) -- no cloud GPU account needed. This mirrors `fastsft --local` / `trial_run.py`.

In [ ]:
from fastsft.helper import current_timestamp
from fastsft.pipeline import DistillationPipeline
from fastsft.progress import log
from fastsft.stages.constants import DATA_GENERATOR, FINE_TUNER

pipeline = DistillationPipeline(
    child_model_id=CHILD_MODEL_ID,
    generation=generation,
    local_training=True,  # train right here on Colab instead of dispatching to Modal
    dataset_repo_id=HF_DATASET_REPO_ID or None,
    model_repo_id=HF_MODEL_REPO_ID or None,
)

run_id = current_timestamp()
raw_dataset_path = None
adapter_dir = None
for stage, output in pipeline.run(PROMPT):
    path = stage.save_output(output, run_id)
    if path:
        log(f"Saved {stage.name} output to '{path}'")
    if stage.name == DATA_GENERATOR:
        raw_dataset_path = path
        # Peek at a couple of generated (instruction, answer) pairs while
        # formatting/training continue -- the next cell has the full viewer.
        sample_rows = list(output["default"]["train"])[:2]
        preview = "\n\n".join(
            f"**Q:** {row['messages'][0]['content']}\n\n**A:** {row['messages'][1]['content'][:400]}..."
            for row in sample_rows
        )
        display(Markdown(f"#### Sample of your generated dataset\n\n{preview}"))
    if stage.name == FINE_TUNER:
        adapter_dir = path  # `path`, not `output` -- the persisted copy, matched by run_id below

log("\nDone! Your adapter is trained.")

### Browse your generated dataset

FastSFT's dataset viewer pretty-prints saved samples -- the same tool you'd run locally as `python -m fastsft.data.viewer`.

In [ ]:
from fastsft.data.viewer import DataViewer

DataViewer(raw_dataset_path).raw_samples(5)

### Evaluate your model

Same judge-scored comparison as Part 1's precomputed example, now run live against your own model and its own real parent teacher. Costs a few OpenRouter calls per prompt (parent answer + judging both ways round) -- keep `EVAL_PROMPTS` short for a quick, cheap check.

In [ ]:
from fastsft.eval.config import EvalConfig
from fastsft.eval.evaluator import Evaluator
from fastsft.eval.prompt_set import load_training_prompts
from fastsft.helper import load_training_metadata

# Pulls the real parent model/style-prompt this adapter was trained against,
# so the reference answers below match the actual teacher -- same as the
# `fastsft-eval` CLI does automatically.
_metadata = load_training_metadata(adapter_dir) or {}

# Small, editable eval set -- automatically drops any that match one of the
# model's own training questions, so this measures generalization, not
# memorization. Each surviving prompt costs three generations (parent, tuned,
# untuned) plus judging both ways round.
_candidate_prompts = [
    "What's the best way to handle a difficult coworker?",
    "Should I move in with my friend?",
    "How do I get better at chess?",
]
_training_questions = {" ".join(q.lower().split()) for q in load_training_prompts(adapter_dir)}
EVAL_PROMPTS = [p for p in _candidate_prompts if " ".join(p.lower().split()) not in _training_questions]

eval_config = EvalConfig(
    adapter_dir=adapter_dir,
    run_id=current_timestamp(),
    parent_model=_metadata.get("parent_model", "meta-llama/llama-3.3-70b-instruct"),
    parent_instruction=_metadata.get("parent_instruction", ""),
    parent_max_tokens=_metadata.get("parent_max_tokens", 1024),
    parent_temperature=_metadata.get("parent_temperature", 0.9),
    num_eval_prompts=len(EVAL_PROMPTS),
)
eval_results = Evaluator(eval_config).run(EVAL_PROMPTS)
render_eval_results(eval_results, adapter_dir)

### Your training loss curve

Same telemetry viewer as Part 1, now for the run you just trained.

In [ ]:
render_stats(load_stats(adapter_dir), adapter_dir)

### Compare your new model

Same side-by-side comparison as Part 1, now against the adapter you just trained.

In [ ]:
from fastsft.eval.prompt_set import load_training_prompts

def _not_in_training(candidates: list[str], adapter_dir: str) -> str:
    """First candidate that wasn't one of adapter_dir's own training questions
    -- keeps this a generalization check, not a memorization check."""
    seen = {" ".join(q.lower().split()) for q in load_training_prompts(adapter_dir)}
    return next(c for c in candidates if " ".join(c.lower().split()) not in seen)

_my_engine = ChildInferenceEngine(adapter_dir)

# Edit these to whatever you want to ask.
compare(_my_engine, _not_in_training([
    "What's the best way to handle a difficult coworker?",
    "How do I get better at chess?",
    "Should I move in with my friend?",
], adapter_dir))

---
## Want more?

This notebook only scratches the surface -- the full [FastSFT repo](https://github.com/AniDas10/FastSFT) has:

- A CLI (`fastsft`) for running the same pipeline from your terminal, with every knob exposed (parent/judge/guide models, LoRA rank, epochs, GPU tier, ...).
- Cloud GPU training via [Modal](https://modal.com) for bigger child models than a free Colab GPU can handle.
- An evaluation suite (`fastsft-eval`) that LLM-judges your tuned model against its untuned base *and* the parent teacher, plus embedding-similarity scoring.
- [`TUTORIAL.md`](https://github.com/AniDas10/FastSFT/blob/main/TUTORIAL.md) and per-stage deep dives on data generation, training, and evaluation.

Built on [distilabel](https://github.com/argilla-io/distilabel), [OpenRouter](https://openrouter.ai), Modal, Hugging Face, and PEFT. Personal project, built for fun to learn how these models work under the hood -- have fun with it.